# Local Brain Tumor Training Notebook

This notebook is rewritten for local use in VS Code.

It supports:
- YOLOv12-style localization training on `backend/brain-tumor`
- optional 14-class tumor classification training from a folder-based dataset
- backend integration with both trained models


In [ ]:
from pathlib import Path
import os

BACKEND_DIR = Path.cwd().resolve()
if BACKEND_DIR.name != "backend":
    candidate = BACKEND_DIR / "backend"
    if candidate.exists():
        BACKEND_DIR = candidate.resolve()

YOLO_DATASET_DIR = BACKEND_DIR / "brain-tumor"
YOLO_NOTEBOOK_YAML = YOLO_DATASET_DIR / "brain-tumor-local.yaml"
YOLO_RUNS_DIR = BACKEND_DIR / "runs"
CLASSIFIER_DATASET_DIR = Path(r"C:\Users\jnthn\projects\BTDS-pixel-master\BTDS-pixel-master\backend\BrainTumor classes")
CLASSIFIER_MODEL_PATH = BACKEND_DIR / "models" / "brain_tumor_14class.keras"
YOLO_BEST_PATH = YOLO_RUNS_DIR / "brain_tumor_train" / "weights" / "best.pt"

CLASSIFIER_LABELS = [
    "astrocytoma",
    "carcinoma",
    "ependymoma",
    "ganglioglioma",
    "germinoma",
    "glioblastoma",
    "granuloma",
    "medulloblastoma",
    "meningioma",
    "neurocytoma",
    "oligodendroglioma",
    "papilloma",
    "schwannoma",
    "tuberculoma",
]

print("BACKEND_DIR:", BACKEND_DIR)
print("YOLO_DATASET_DIR:", YOLO_DATASET_DIR)
print("CLASSIFIER_DATASET_DIR:", CLASSIFIER_DATASET_DIR)
print("CLASSIFIER_MODEL_PATH:", CLASSIFIER_MODEL_PATH)
print("YOLO_BEST_PATH:", YOLO_BEST_PATH)


In [ ]:
# Run this once in the notebook kernel if the packages are missing.
# Restart the kernel after installation if VS Code asks you to.

%pip install -r requirements.txt
%pip install -r requirements-yolo.txt


In [ ]:
from collections import Counter

required_dirs = [
    YOLO_DATASET_DIR / "images" / "train",
    YOLO_DATASET_DIR / "images" / "val",
    YOLO_DATASET_DIR / "labels" / "train",
    YOLO_DATASET_DIR / "labels" / "val",
]

missing_dirs = [str(path) for path in required_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError("Missing YOLO dataset folders:\n" + "\n".join(missing_dirs))

train_images = {p.stem for p in (YOLO_DATASET_DIR / "images" / "train").glob("*") if p.is_file()}
val_images = {p.stem for p in (YOLO_DATASET_DIR / "images" / "val").glob("*") if p.is_file()}
train_labels = {p.stem for p in (YOLO_DATASET_DIR / "labels" / "train").glob("*.txt")}
val_labels = {p.stem for p in (YOLO_DATASET_DIR / "labels" / "val").glob("*.txt")}

summary = {
    "train_images": len(train_images),
    "train_labels": len(train_labels),
    "train_backgrounds": len(train_images - train_labels),
    "val_images": len(val_images),
    "val_labels": len(val_labels),
    "val_backgrounds": len(val_images - val_labels),
    "extra_train_labels": len(train_labels - train_images),
    "extra_val_labels": len(val_labels - val_images),
}

if summary["extra_train_labels"] or summary["extra_val_labels"]:
    raise ValueError(summary)

summary


In [ ]:
import yaml

yolo_config = {
    "path": YOLO_DATASET_DIR.as_posix(),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "negative",
        1: "positive",
    },
}

YOLO_NOTEBOOK_YAML.write_text(yaml.safe_dump(yolo_config, sort_keys=False), encoding="utf-8")
print(YOLO_NOTEBOOK_YAML.read_text(encoding="utf-8"))


In [ ]:
from ultralytics import YOLO

# Use `device=0` if you have a CUDA GPU configured correctly.
device = "cpu"

# Try a pretrained checkpoint first. If it is not available locally or cannot be fetched,
# fall back to the built-in YAML model definition from the installed Ultralytics package.
primary_model_name = "yolo11n.pt"
fallback_model_name = "yolo11n.yaml"

try:
    model = YOLO(primary_model_name)
    selected_model_name = primary_model_name
except FileNotFoundError:
    model = YOLO(fallback_model_name)
    selected_model_name = fallback_model_name

print("Training with model:", selected_model_name)
train_results = model.train(
    data=str(YOLO_NOTEBOOK_YAML),
    epochs=20,
    batch=16,
    imgsz=640,
    workers=2,
    device=device,
    project=str(YOLO_RUNS_DIR),
    name="brain_tumor_train",
    exist_ok=True,
)

train_results


In [ ]:
from ultralytics import YOLO

trained_detector = YOLO(str(YOLO_BEST_PATH))
sample_images = sorted((YOLO_DATASET_DIR / "images" / "val").glob("*.jpg"))[:3]
sample_images = sample_images or sorted((YOLO_DATASET_DIR / "images" / "val").glob("*.png"))[:3]

if not sample_images:
    raise FileNotFoundError("No validation images found for preview.")

detector_results = trained_detector([str(path) for path in sample_images], verbose=False)
for item, image_path in zip(detector_results, sample_images):
    print(image_path.name, "boxes=", 0 if item.boxes is None else len(item.boxes))
    display(item.plot())


## Optional: 14-class tumor classifier

Your 14-class dataset should be folder-based, for example:

```text
brain-tumor-14-classes/
  astrocytoma/
  carcinoma/
  ependymoma/
  ...
  tuberculoma/
```

This model works together with YOLO: YOLO localizes the tumor, then the classifier predicts the tumor type from the crop.


In [ ]:
if not CLASSIFIER_DATASET_DIR.exists():
    raise FileNotFoundError(
        "Set CLASSIFIER_DATASET_DIR to your 14-class dataset folder before running this cell."
    )

# Notebook-friendly direct training block.
import sys

try:
    import tensorflow as tf
except ModuleNotFoundError as exc:
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    raise RuntimeError(
        "TensorFlow is not available in this notebook kernel. "
        f"Current interpreter: Python {version}. "
        "Use a Python 3.10 or 3.11 environment with a working TensorFlow install, then rerun this cell."
    ) from exc

img_size = 224
batch_size = 16
epochs = 15

train_ds = tf.keras.utils.image_dataset_from_directory(
    CLASSIFIER_DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_size, img_size),
    batch_size=batch_size,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    CLASSIFIER_DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_size, img_size),
    batch_size=batch_size,
)

class_names = train_ds.class_names
print(class_names)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=autotune)
val_ds = val_ds.cache().prefetch(buffer_size=autotune)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(img_size, img_size, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(img_size, img_size, 3))
x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
classifier_model = tf.keras.Model(inputs, outputs)

classifier_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = classifier_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
    ],
)

CLASSIFIER_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
classifier_model.save(CLASSIFIER_MODEL_PATH)
(CLASSIFIER_MODEL_PATH.with_suffix(".labels.txt")).write_text("\n".join(class_names) + "\n", encoding="utf-8")

print("Saved classifier:", CLASSIFIER_MODEL_PATH)
print("Saved labels:", CLASSIFIER_MODEL_PATH.with_suffix(".labels.txt"))


In [ ]:
# Backend integration values.
# Use these in your VS Code terminal before running the Flask app.

print("PowerShell:")
print(f'$env:YOLO_MODEL_PATH="{YOLO_BEST_PATH}"')
print(f'$env:CLASSIFIER_MODEL_PATH="{CLASSIFIER_MODEL_PATH}"')
print('$env:CLASSIFIER_LABELS="' + ','.join(CLASSIFIER_LABELS) + '"')
print('$env:CLASSIFIER_IMAGE_SIZE="224"')
